# Clasificación: riesgo de cancelación con Random Forest

Este notebook utiliza las colecciones `pedidos`, `usuarios` y `productos`. Cada fila representa **un pedido completo**. La variable objetivo es `clase_y`: **Cancelado = 1** y **Entregado = 0**. Los cálculos existen únicamente en el DataFrame; no se agregan campos a MongoDB.

## 1. Dependencias
Si hace falta, instala: `pip install pymongo python-dotenv pandas scikit-learn matplotlib seaborn joblib`.

In [ ]:
import os
from pathlib import Path
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score)
from sklearn.pipeline import Pipeline

env_path = next((p for p in [Path.cwd() / '.env', Path.cwd().parent / '.env'] if p.exists()), None)
load_dotenv(env_path)
MONGO_URI = os.getenv('MONGO_URI')
if not MONGO_URI:
    raise RuntimeError('No se encontró MONGO_URI. Abre Jupyter desde pryBinaBack o carga el archivo .env.')

## 2. Extracción de todo el historial finalizado
Se utilizan todos los pedidos `Entregado` y `Cancelado`; no se seleccionan vecinos ni una muestra de pedidos similares.

In [ ]:
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000)
db = client.get_default_database()
pedidos = list(db.pedidos.find(
    {'estado': {'$in': ['Entregado', 'Cancelado']}},
    {'usuario': 1, 'productos': 1, 'total': 1, 'metodoPago': 1,
     'estado': 1, 'costoEnvio': 1, 'createdAt': 1}
).sort('createdAt', 1))
usuarios = {str(u['_id']): u for u in db.usuarios.find(
    {}, {'nombre': 1, 'ap': 1, 'am': 1, 'email': 1, 'fechaNacimiento': 1}
)}
print('Pedidos finalizados:', len(pedidos))
print('Usuarios encontrados:', len(usuarios))

## 3. Transformación: una fila por pedido
Las ocho variables conceptuales son: `edad`, `producto`, `cantidad`, `precio`, `total`, `costo_envio`, `metodo_pago` y `porcentaje_cancelados_previos`. Para conservar varios productos sin promediar, cada producto genera internamente dos columnas: `cantidad_producto_<id>` y `precio_producto_<id>`.

In [ ]:
def calcular_edad(fecha_nacimiento, fecha_referencia):
    if not fecha_nacimiento or not fecha_referencia:
        return 0
    return fecha_referencia.year - fecha_nacimiento.year - (
        (fecha_referencia.month, fecha_referencia.day) <
        (fecha_nacimiento.month, fecha_nacimiento.day)
    )

def construir_fila(pedido, usuario, porcentaje_previo):
    fila = {
        'edad': calcular_edad(usuario.get('fechaNacimiento'), pedido.get('createdAt')),
        'total': float(pedido.get('total', 0) or 0),
        'costo_envio': float(pedido.get('costoEnvio', 0) or 0),
        'porcentaje_cancelados_previos': float(porcentaje_previo),
        'metodo_pago': pedido.get('metodoPago') or 'Sin definir'
    }
    for item in pedido.get('productos', []):
        producto_id = str(item.get('producto', ''))
        if not producto_id:
            continue
        clave_cantidad = f'cantidad_producto_{producto_id}'
        clave_precio = f'precio_producto_{producto_id}'
        fila[clave_cantidad] = fila.get(clave_cantidad, 0) + float(item.get('cantidad', 0) or 0)
        fila[clave_precio] = float(item.get('precio', 0) or 0)
    return fila

registros, X_dict, y, historial = [], [], [], {}
for pedido in pedidos:
    usuario_id = str(pedido.get('usuario', ''))
    usuario = usuarios.get(usuario_id, {})
    previos, cancelados = historial.get(usuario_id, (0, 0))
    porcentaje_previo = (cancelados / previos * 100) if previos else 0
    X_dict.append(construir_fila(pedido, usuario, porcentaje_previo))
    clase = 1 if pedido.get('estado') == 'Cancelado' else 0
    y.append(clase)
    registros.append({
        'pedido_id': str(pedido['_id']),
        'usuario': usuario_id,
        'nombre_usuario': ' '.join(filter(None, [usuario.get('nombre'), usuario.get('ap'), usuario.get('am')])),
        'fecha': pedido.get('createdAt'),
        'numero_productos': len(pedido.get('productos', [])),
        'porcentaje_cancelados_previos': porcentaje_previo,
        'clase_y': clase
    })
    historial[usuario_id] = (previos + 1, cancelados + clase)

df_contexto = pd.DataFrame(registros)
print('Filas del dataset / pedidos:', len(df_contexto))
display(df_contexto.head())
display(df_contexto['clase_y'].value_counts().rename({0: 'Entregado', 1: 'Cancelado'}))

## 4. Separación cronológica para evaluación
El 80% más antiguo se usa para entrenamiento y el 20% más reciente para prueba. Después de evaluar, se entrena el modelo final con todos los pedidos finalizados.

In [ ]:
corte = max(1, min(len(X_dict) - 1, int(len(X_dict) * 0.80)))
X_train_dict, X_test_dict = X_dict[:corte], X_dict[corte:]
y_train, y_test = y[:corte], y[corte:]
print('Pedidos de entrenamiento:', len(X_train_dict))
print('Pedidos de prueba:', len(X_test_dict))

modelo_evaluacion = Pipeline([
    ('vectorizacion', DictVectorizer(sparse=True)),
    ('clasificador', RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=20,
        min_samples_leaf=8, class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
modelo_evaluacion.fit(X_train_dict, y_train)

## 5. Evaluación del modelo

In [ ]:
probabilidad = modelo_evaluacion.predict_proba(X_test_dict)[:, 1]
prediccion = (probabilidad >= 0.50).astype(int)
print('Accuracy:', round(accuracy_score(y_test, prediccion), 3))
print('Precision:', round(precision_score(y_test, prediccion, zero_division=0), 3))
print('Recall:', round(recall_score(y_test, prediccion, zero_division=0), 3))
print('F1:', round(f1_score(y_test, prediccion, zero_division=0), 3))
if len(set(y_test)) == 2:
    print('ROC-AUC:', round(roc_auc_score(y_test, probabilidad), 3))
print(classification_report(y_test, prediccion, target_names=['Entregado', 'Cancelado'], zero_division=0))
cm = confusion_matrix(y_test, prediccion)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Entregado', 'Cancelado'],
            yticklabels=['Entregado', 'Cancelado'])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de confusión - Random Forest')
plt.show()

## 6. Variables con mayor importancia
Esta gráfica ayuda a explicar qué datos influyeron más en el bosque. No significa causalidad.

In [ ]:
vectorizador = modelo_evaluacion.named_steps['vectorizacion']
bosque = modelo_evaluacion.named_steps['clasificador']
importancias = pd.Series(bosque.feature_importances_, index=vectorizador.get_feature_names_out())
display(importancias.sort_values(ascending=False).head(20).to_frame('importancia'))

## 7. Modelo final con todo el dataset y predicción de pendientes
La evaluación anterior separó datos para medir el desempeño. Ahora el modelo definitivo aprende de todos los pedidos cuyo resultado ya se conoce y recibe automáticamente los pedidos `Pendiente`.

In [ ]:
modelo_final = Pipeline([
    ('vectorizacion', DictVectorizer(sparse=True)),
    ('clasificador', RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=20,
        min_samples_leaf=8, class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
modelo_final.fit(X_dict, y)

pendientes = list(db.pedidos.find({'estado': 'Pendiente'}).sort('createdAt', -1))
X_pendientes, contexto_pendientes = [], []
for pedido in pendientes:
    usuario_id = str(pedido.get('usuario', ''))
    usuario = usuarios.get(usuario_id, {})
    previos, cancelados = historial.get(usuario_id, (0, 0))
    porcentaje_previo = (cancelados / previos * 100) if previos else 0
    X_pendientes.append(construir_fila(pedido, usuario, porcentaje_previo))
    contexto_pendientes.append({
        'pedido_id': str(pedido['_id']),
        'usuario': usuario_id,
        'nombre_usuario': ' '.join(filter(None, [usuario.get('nombre'), usuario.get('ap'), usuario.get('am')])),
        'email': usuario.get('email', ''),
        'total': pedido.get('total', 0),
        'porcentaje_cancelados_previos': porcentaje_previo
    })

if not X_pendientes:
    print('No hay pedidos Pendiente para clasificar.')
else:
    resultado = pd.DataFrame(contexto_pendientes)
    resultado['probabilidad_cancelacion'] = modelo_final.predict_proba(X_pendientes)[:, 1]
    resultado['porcentaje_riesgo'] = (resultado['probabilidad_cancelacion'] * 100).round(1)
    resultado['nivel'] = pd.cut(
        resultado['probabilidad_cancelacion'], [-1, .35, .60, 1],
        labels=['Bajo', 'Medio', 'Alto'], include_lowest=True
    )
    display(resultado.sort_values('probabilidad_cancelacion', ascending=False))

In [ ]:
joblib.dump(modelo_final, 'modelo_random_forest_cancelacion.joblib')
print('Modelo guardado como modelo_random_forest_cancelacion.joblib')
client.close()